# Prepare the ERA5 files for the downscaling notebook

Cuts hourly ERA5 convective and large-scale rain for each typhoon from Google's public ARCO-ERA5 copy
(no account needed) and saves one file per typhoon, a few MB each. Run once, then commit the files to `data/`.
The participant notebook downloads them from there.

To add a typhoon, add a line to `CASES` and a matching entry to `TYPHOONS` in the participant notebook.

In [ ]:
%pip install -q gcsfs zarr
import xarray as xr

ds = xr.open_zarr("gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3",
                  chunks=None, storage_options=dict(token="anon"))
print("ERA5 available until:", ds.attrs.get("valid_time_stop"))

In [ ]:
CASES = {  # file: first and last hour (UTC). Each file covers 4 to 26 N, 112 to 132 E.
    "nando_era5.nc": ("2025-09-20T00", "2025-09-24T00"),
    "kristine_era5.nc": ("2024-10-21T00", "2024-10-25T00"),
    "tino_era5.nc": ("2025-11-02T00", "2025-11-06T00"),
}

for name, (start, end) in CASES.items():
    sub = (ds[["convective_precipitation", "large_scale_precipitation"]]
           .sel(time=slice(start, end), latitude=slice(26, 4), longitude=slice(112, 132))
           .rename(convective_precipitation="cp", large_scale_precipitation="lsp", latitude="lat", longitude="lon")
           .load())
    assert sub.sizes["time"] == 97 and int(sub.isnull().sum().to_array().sum()) == 0
    enc = {v: {"zlib": True, "complevel": 4, "dtype": "float32"} for v in sub.data_vars}
    sub.to_netcdf(name, encoding=enc)
    print(f"{name}: max hourly rain {float((sub.cp + sub.lsp).max()) * 1000:.1f} mm")

In [ ]:
import zipfile
from google.colab import files

with zipfile.ZipFile("typhoon_era5.zip", "w") as z:
    for name in CASES:
        z.write(name)
files.download("typhoon_era5.zip")